## Part A: Conceptual Foundation (Theory)


**Q1. What is Regularization in Machine Learning? Why is it needed?**

Regularization adds a penalty term to the model's loss function to discourage overly complex models.
Without regularization, a model can memorize training data (overfit) and fail on new data.
It forces the model to keep weights small, improving generalization.

**Q2. Difference between Ridge Regression (L2) and Lasso Regression (L1)**

| Feature | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Penalty | Sum of squared coefficients | Sum of absolute coefficients |
| Effect on weights | Shrinks all weights toward zero | Can set some weights exactly to zero |
| Feature Selection | No — keeps all features | Yes — performs automatic feature selection |
| Best use | When all features matter | When only a few features matter |

**Q3. What is Cross-Validation and why is it important?**

Cross-validation evaluates model performance by splitting data into multiple train/test folds.
It gives a reliable estimate of how the model will perform on unseen data, and helps tune hyperparameters without overfitting to a single test set.

**Q4. Cross-Validation Techniques**

- **K-Fold**: Data split into K parts. Model trains on K-1 parts, tests on 1. Repeated K times.
- **Stratified K-Fold**: Like K-Fold but preserves the distribution of target values in each fold. Useful for imbalanced data.
- **LOOCV (Leave-One-Out)**: Each sample is its own test set. Very thorough but slow on large datasets.
- **Time Series Split**: Respects time order — training always uses past data, test uses future data. Prevents data leakage.

**Q5. Why are tree-based models less sensitive to feature scaling?**

Trees split on thresholds, not distances or dot products. Whether `area` is in sq-ft or sq-meters, the best split threshold just shifts — the model logic doesn't change. Linear models and SVMs use distances/magnitudes directly, so they're affected by scale.


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso,LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

---
## Part B: Dataset Understanding & Preparation

In [3]:
df = pd.read_csv('Advanced_Regression_HousePrice_Dataset.csv')
print("Shape:", df.shape)
df.head()

Shape: (3800, 12)


,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


In [6]:
print(df.dtypes)
print(df.isnull().sum())
df.describe()

property_id           int64
sale_date            object
area_sqft             int64
bedrooms              int64
bathrooms             int64
location_score      float64
property_age          int64
distance_city_km    float64
near_school           int64
near_metro            int64
crime_rate_index    float64
house_price_inr       int64
dtype: object
property_id         0
sale_date           0
area_sqft           0
bedrooms            0
bathrooms           0
location_score      0
property_age        0
distance_city_km    0
near_school         0
near_metro          0
crime_rate_index    0
house_price_inr     0
dtype: int64


,property_id,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
count,3800.00000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3.800000e+03
mean,201900.50000,1716.925526,3.428158,2.916316,6.502237,22.537105,13.085132,0.548421,0.472895,4.242911,2.071940e+07
std,1097.10984,582.996559,1.356682,1.133540,1.766945,12.325740,6.537425,0.497715,0.499330,2.045371,8.707465e+06
min,200001.00000,500.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.500000,1.506126e+06
25%,200950.75000,1322.000000,2.000000,2.000000,5.300000,14.000000,8.500000,0.000000,0.000000,2.810000,1.446895e+07
50%,201900.50000,1700.500000,3.000000,3.000000,6.500000,20.000000,13.000000,1.000000,0.000000,4.220000,1.989180e+07
75%,202850.25000,2105.000000,4.000000,4.000000,7.700000,29.000000,17.500000,1.000000,1.000000,5.650000,2.596062e+07
max,203800.00000,3776.000000,7.000000,6.000000,10.000000,80.000000,38.700000,1.000000,1.000000,12.000000,5.930315e+07


In [7]:
# non-predictive columns
df = df.drop(columns=["property_id","sale_date"])

# now input features
X = df.drop(columns="house_price_inr")

# target variable
y = df["house_price_inr"]

print(f"input features:-{X.columns.tolist()}")
print("target variable :- house_price_inr")
print(X.shape)

input features:-['area_sqft', 'bedrooms', 'bathrooms', 'location_score', 'property_age', 'distance_city_km', 'near_school', 'near_metro', 'crime_rate_index']
target variable :- house_price_inr
(3800, 9)


In [11]:
# now train-test-split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=18)
print(f"train data: {X_train.shape},test data: {X_test.shape}")

train data: (3040, 9),test data: (760, 9)


In [12]:
# now basic scaling through standard scaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## Part C: Regularized Linear Models (Ridge & Lasso)

In [41]:
# a user-defined function to fit and evaluate model

all_results = []

def test_evaluate_model(name,model,X_tr,X_te,y_tr,y_te):
    model.fit(X_tr,y_tr)
    y_pred = model.predict(X_te)
    mse = mean_squared_error(y_te,y_pred)
    mae = mean_absolute_error(y_te,y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te,y_pred)
    
    print(f"model_name:{name}")
    print(f"MSE:{mse}")
    print(f"MAE:{mae}")
    print(f"RMSE:{rmse}")
    print(f"r2_score:{r2}")
    
    return {'Model': name, 'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MSE': mse}
    

In [43]:
# ridge regression L2

ridge_model = Ridge(alpha=1.0)
ridge_L2 = test_evaluate_model("Ridge Regression (alpha=1)", ridge_model,X_train_scaled, X_test_scaled, y_train, y_test)
all_results.append(ridge_L2)

model_name:Ridge Regression (alpha=1)
MSE:5976709372382.629
MAE:1887600.5314399863
RMSE:2444730.940693194
r2_score:0.9184124442211243


In [46]:
# lasso regression L1

lasso_model = Lasso(alpha=1.0, max_iter=10000)
Lasso_L1 = test_evaluate_model("Lasso Regression (alpha=1)", lasso_model,X_train_scaled, X_test_scaled, y_train, y_test)
all_results.append(Lasso_L1)

model_name:Lasso Regression (alpha=1)
MSE:5978518761419.219
MAE:1888123.8687129435
RMSE:2445100.97162044
r2_score:0.9183877444039255
